# PyTorch 第十一章：ONNX 使用

> 对应《PyTorch 实用教程（第二版）》第十一章  
> 目标：理解 **PyTorch → ONNX → ONNX Runtime** 的完整部署链路，并掌握动态形状、Execution Provider、性能评测、量化、图优化、线程管理、I/O Binding 与 profiling。

## 原教程小节顺序

1. 11.1 ONNX 简介与安装
2. 11.2 ONNX Runtime 简介与使用
3. 11.3 ONNX Runtime 进阶使用

教程章节入口：  
https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-11/

## 本 Notebook 的取舍

- 不导出 ResNet50 等大型模型，使用小型 MLP / CNN 等价演示。
- 在 Google Colab 中可自动安装 `onnx`、`onnxruntime`、`onnxscript`。
- 在没有联网或没有 ONNX 依赖的本地校验环境中，会自动跳过 ONNX 专属实验，而不会中断 Notebook。
- 重点使用当前 PyTorch 推荐的 `dynamo=True` ONNX exporter。
- 保留旧版 `dynamic_axes` 概念，但明确当前更推荐 `dynamic_shapes`。

# 0. 环境导入

In [ ]:
import importlib
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("device:", device)

## 0.1 ONNX 相关依赖

需要区分：

- `onnx`：模型格式、计算图与算子定义；
- `onnxruntime`：运行 ONNX 模型的推理引擎；
- `onnxscript`：当前 PyTorch 新 ONNX exporter 依赖之一。

这三个包不是一回事。

In [ ]:
def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IN_COLAB = running_in_colab()

required = ["onnx", "onnxruntime", "onnxscript"]
missing = []

for name in required:
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(name)

if missing and IN_COLAB:
    print("Colab detected. Installing:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

# 重新检测
HAS_ONNX = True
try:
    import onnx
    import onnxruntime as ort
except ImportError:
    HAS_ONNX = False
    onnx = None
    ort = None

print("IN_COLAB:", IN_COLAB)
print("HAS_ONNX:", HAS_ONNX)

if not HAS_ONNX:
    print(
        "当前校验环境未安装 ONNX 依赖，因此 ONNX 专属单元会安全跳过；"
        "在 Google Colab 中本单元会自动安装依赖后继续运行。"
    )

# 11.1 ONNX 简介与安装

原教程顺序：

1. 为什么需要 ONNX
2. ONNX 基础概念
3. ONNX operator / opset
4. 安装 ONNX
5. PyTorch 导出 ONNX
6. 固定 shape 与动态 shape

## 为什么需要中间格式？

假设有：

- $N$ 个训练框架
- $M$ 个推理后端

如果每一对都单独适配，接口数量接近：

$$
O(NM)
$$

如果大家都适配一个中间交换格式，则变为：

$$
O(N+M)
$$

ONNX 的核心价值就是把**训练框架**与**推理后端**解耦。

## ONNX 文件本质：计算图，而不是 Python 源码

ONNX 模型通常包含：

- **Graph**：整个计算图
- **Node**：算子节点，例如 `MatMul`、`Add`、`Conv`
- **Input / Output**：图输入输出
- **Initializer**：权重等常量
- **Attribute**：算子属性
- **Opset**：算子语义版本

部署时真正运行的是这张图，而不是原来的 `nn.Module.forward()` Python 代码。

## 11.1.1 手工理解一个线性层的计算图

线性层：

$$
Y = XW^T + b
$$

可以拆成两个核心算子：

```text
X ── MatMul ── Add ── Y
       ↑         ↑
       W         b
```

真实导出器会负责把 PyTorch 运算映射到 ONNX 算子。

In [ ]:
# 先用纯 PyTorch 验证数学关系
X = torch.tensor([[1.0, 2.0]])
W = torch.tensor([[3.0, 4.0],
                  [5.0, 6.0]])
b = torch.tensor([0.5, -0.5])

Y = X @ W.T + b

print("X shape:", X.shape)
print("W shape:", W.shape)
print("Y:", Y)

## 11.1.2 Operator 与 Opset

最容易混淆：

- **ONNX version**：ONNX 格式 / 库版本；
- **opset version**：一组 ONNX 算子的语义版本。

部署失败常见原因之一：

> 导出的算子或 opset 超出了目标推理后端支持范围。

因此不要机械追求“最高 opset”，而应根据部署后端支持情况选择。

In [ ]:
if HAS_ONNX:
    print("onnx package version:", onnx.__version__)
    print("ONNX supported/latest opset in installed package:",
          onnx.defs.onnx_opset_version())
else:
    print("SKIP: install onnx to inspect local opset support.")

## 11.1.3 构造一个最小待部署模型

模型故意很小，便于检查：

`Linear → ReLU → Linear`

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
        )

    def forward(self, x):
        return self.net(x)

model = TinyNet().eval()
example = torch.randn(2, 4)

with torch.inference_mode():
    torch_output = model(example)

print(model)
print("input:", example.shape)
print("output:", torch_output.shape)

## 11.1.4 当前推荐的 PyTorch ONNX 导出方式

当前 PyTorch 推荐基于 `torch.export` 的 ONNX exporter：

```python
torch.onnx.export(
    model,
    (example_input,),
    "model.onnx",
    dynamo=True,
)
```

相较旧教程：

- 旧主路径：TorchScript / tracing exporter；
- 当前主路径：`torch.export` + `dynamo=True`；
- 当前动态图形优先使用 `dynamic_shapes`；
- `dynamic_axes` 主要保留给旧 exporter / 兼容代码。

另外，大模型超过 ONNX 单文件大小限制时，当前 exporter 还支持 external data。

In [ ]:
onnx_path = Path("/tmp/tiny_net.onnx")

if HAS_ONNX:
    onnx_program = torch.onnx.export(
        model,
        (example,),
        dynamo=True,
        input_names=["input"],
        output_names=["logits"],
    )
    onnx_program.save(onnx_path)
    print("saved:", onnx_path)
    print("exists:", onnx_path.exists())
else:
    print("SKIP: ONNX dependencies unavailable in this validation environment.")

## 11.1.5 检查 ONNX 模型是否合法

推荐导出后至少做三件事：

1. `onnx.load()`
2. `onnx.checker.check_model()`
3. 用 ONNX Runtime 与 PyTorch 做数值对齐

只“成功生成 .onnx 文件”并不代表部署正确。

In [ ]:
if HAS_ONNX and onnx_path.exists():
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    print("graph name:", onnx_model.graph.name)
    print("num nodes:", len(onnx_model.graph.node))
    print("inputs:", [x.name for x in onnx_model.graph.input])
    print("outputs:", [x.name for x in onnx_model.graph.output])
    print("ONNX checker: PASSED")
else:
    print("SKIP")

## 11.1.6 查看计算图里的算子

Netron 很适合视觉检查 `.onnx` 文件，但程序内也可以直接读 node。

In [ ]:
if HAS_ONNX and onnx_path.exists():
    op_types = [node.op_type for node in onnx_model.graph.node]
    print("operators:", op_types)
else:
    print("SKIP")

## 11.1.7 固定 shape vs 动态 shape

默认导出器会根据 example input 推断 shape。

如果只用 `[2, 4]` 导出：

- batch=2 可能被固定；
- 实际部署时 batch=1、8 等可能不能直接接受。

现代 exporter 更推荐 `dynamic_shapes`。

In [ ]:
dynamic_onnx_path = Path("/tmp/tiny_net_dynamic.onnx")

if HAS_ONNX:
    dynamic_program = torch.onnx.export(
        model,
        (example,),
        dynamo=True,
        input_names=["input"],
        output_names=["logits"],
        dynamic_shapes=({0: "batch"},),
    )
    dynamic_program.save(dynamic_onnx_path)
    print("saved dynamic model:", dynamic_onnx_path)
else:
    print("SKIP")

旧教程中的写法：

```python
dynamic_axes={
    "input": {0: "batch"},
    "output": {0: "batch"},
}
```

今天仍能在 `dynamo=False` 的旧 exporter 路径中见到，但新 exporter 下应优先使用 `dynamic_shapes`。

# 11.2 ONNX Runtime 简介与使用

原教程顺序：

1. ONNX Runtime 与 ONNX 的区别
2. CPU / GPU 安装
3. `InferenceSession`
4. Execution Provider
5. 输入输出名称
6. 推理
7. latency / throughput
8. batch size 性能权衡

## 11.2.1 ONNX ≠ ONNX Runtime

- ONNX：模型交换格式。
- ONNX Runtime（ORT）：执行 ONNX 图的推理引擎。

类比：

```text
.onnx 文件 ≈ 程序
ONNX Runtime ≈ 执行程序的运行时
```

## 11.2.2 Execution Provider（EP）

ORT 通过 Execution Provider 把算子交给不同硬件后端，例如：

- `CPUExecutionProvider`
- `CUDAExecutionProvider`
- TensorRT EP
- OpenVINO EP
- CoreML EP
- DirectML EP
- QNN EP

Provider 列表的顺序代表优先级。

当前环境可用 provider：

In [ ]:
if HAS_ONNX:
    print(ort.get_available_providers())
else:
    print("SKIP")

## 11.2.3 创建 InferenceSession

生产代码不要默认假设 GPU provider 一定存在。

更稳妥：

1. 检查 `get_available_providers()`；
2. CUDA 存在时优先 CUDA；
3. 保留 CPU fallback。

In [ ]:
if HAS_ONNX and dynamic_onnx_path.exists():
    available = ort.get_available_providers()

    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in available
        else ["CPUExecutionProvider"]
    )

    session = ort.InferenceSession(
        str(dynamic_onnx_path),
        providers=providers,
    )

    print("session providers:", session.get_providers())
else:
    session = None
    print("SKIP")

## 11.2.4 ORT 主要通过名字传递输入

不要猜 input / output name，直接从 session 查询。

In [ ]:
if session is not None:
    for item in session.get_inputs():
        print("input:", item.name, item.shape, item.type)

    for item in session.get_outputs():
        print("output:", item.name, item.shape, item.type)
else:
    print("SKIP")

## 11.2.5 ONNX Runtime 推理

CPU 上最常见输入形式是 NumPy：

```python
outputs = session.run(
    ["logits"],
    {"input": numpy_array},
)
```

In [ ]:
if session is not None:
    x_np = example.numpy().astype(np.float32)

    ort_output = session.run(
        ["logits"],
        {"input": x_np},
    )[0]

    print("ORT output shape:", ort_output.shape)
else:
    ort_output = None
    print("SKIP")

## 11.2.6 最关键检查：PyTorch 与 ONNX Runtime 数值对齐

部署验证至少应比较：

- shape
- dtype
- 数值误差
- 最终业务结果

浮点运算顺序和 kernel 可能不同，因此不要要求 bitwise 完全一致。

In [ ]:
if ort_output is not None:
    with torch.inference_mode():
        pt = model(example).numpy()

    max_abs_error = np.max(np.abs(pt - ort_output))

    print("max abs error:", max_abs_error)
    np.testing.assert_allclose(pt, ort_output, rtol=1e-4, atol=1e-5)
    print("PyTorch vs ORT: PASSED")
else:
    print("SKIP")

## 11.2.7 验证动态 batch

这比只看 ONNX 文件中的 symbolic shape 更可靠。

In [ ]:
if session is not None:
    for batch_size in [1, 3, 8]:
        x = np.random.randn(batch_size, 4).astype(np.float32)
        y = session.run(None, {"input": x})[0]
        print(f"batch={batch_size}: {x.shape} -> {y.shape}")
else:
    print("SKIP")

## 11.2.8 Latency 与 Throughput 必须分开

### Latency

一次请求等多久：

$$
\text{latency}
=
\frac{\text{total time}}{\text{number of runs}}
$$

### Throughput

单位时间处理多少样本：

$$
\text{throughput}
=
\frac{\text{number of samples}}{\text{total time}}
$$

batch 增大通常可以提高吞吐，但可能提高单个请求延迟。

因此：

> “最快模型”必须说明你优化的是 latency 还是 throughput。

In [ ]:
def benchmark_callable(fn, batch_size, warmup=20, runs=100):
    for _ in range(warmup):
        fn()

    start = time.perf_counter()
    for _ in range(runs):
        fn()
    elapsed = time.perf_counter() - start

    latency_ms = elapsed / runs * 1000
    throughput = batch_size * runs / elapsed
    return latency_ms, throughput

print("benchmark helper ready")

## 11.2.9 用小模型观察 batch size 趋势

注意：这是当前运行环境的小模型实验，不应把结果外推到 GPU、ResNet50 或生产服务。

In [ ]:
if session is not None:
    for bs in [1, 4, 16, 64]:
        x = np.random.randn(bs, 4).astype(np.float32)

        def run_ort():
            session.run(None, {"input": x})

        latency, throughput = benchmark_callable(
            run_ort,
            batch_size=bs,
            warmup=5,
            runs=30,
        )

        print(
            f"bs={bs:>2} | "
            f"latency={latency:.3f} ms | "
            f"throughput={throughput:.1f} samples/s"
        )
else:
    print("SKIP")

### Benchmark 的正确姿势

至少控制这些变量：

- warm-up 次数；
- 测量次数；
- batch size；
- 输入 shape；
- dtype；
- provider；
- CPU/GPU 同步；
- 线程数；
- 是否包含数据预处理与 H2D / D2H 拷贝。

只测一次 `time.time()` 几乎没有意义。

# 11.3 ONNX Runtime 进阶使用

原教程依次介绍：

1. FP16
2. INT8
3. 混合精度
4. 计算图优化
5. 线程管理
6. I/O Binding
7. Profiling

这些技术分别从：

- 数值精度
- 图结构
- CPU 调度
- 数据传输
- 性能诊断

不同层面优化推理。

## 11.3.1 FP16：先分清“精度”与“量化”

严格术语上：

- FP32 → FP16 更常称 **reduced precision / half precision**
- INT8 才是典型整数 quantization

教程把 FP16 也归在“量化策略”中，工程语境能理解，但概念上最好区分。

理论参数存储：

$$
FP32: 4\ bytes
$$

$$
FP16: 2\ bytes
$$

因此仅看权重，理论上可减半。

In [ ]:
num_params = sum(p.numel() for p in model.parameters())

fp32_bytes = num_params * 4
fp16_bytes = num_params * 2
int8_bytes = num_params * 1

print("parameters:", num_params)
print("FP32 theoretical:", fp32_bytes, "bytes")
print("FP16 theoretical:", fp16_bytes, "bytes")
print("INT8 theoretical:", int8_bytes, "bytes")

### FP16 不是“无条件加速”

是否更快取决于：

- GPU / CPU 硬件；
- kernel 支持；
- 算子；
- 内存带宽；
- 转换开销；
- 模型尺寸。

因此必须 benchmark，不要只根据位宽推断速度。

## 11.3.2 INT8 量化基本公式

线性量化通常可以写为：

$$
x_{real}
\approx
scale \cdot (x_q-zero\_point)
$$

量化误差来自把连续浮点值映射到有限整数集合。

In [ ]:
def symmetric_quantize_int8(x):
    x = torch.as_tensor(x, dtype=torch.float32)
    scale = x.abs().max() / 127

    if scale == 0:
        q = torch.zeros_like(x, dtype=torch.int8)
        return q, torch.tensor(1.0), torch.zeros_like(x)

    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)
    x_hat = q.float() * scale
    return q, scale, x_hat

x = torch.randn(1000)
q, scale, x_hat = symmetric_quantize_int8(x)

print("scale:", round(float(scale), 6))
print("mean abs error:", round(float((x - x_hat).abs().mean()), 6))

## Dynamic vs Static Quantization

### Dynamic

- 权重提前量化；
- activation 的量化参数通常在运行时计算；
- 配置简单；
- 常用于 RNN / Transformer / MatMul 为主的模型。

### Static

- 先用 calibration dataset 跑一遍；
- activation 的 scale / zero point 提前估计；
- 常用于 CNN；
- 需要代表性校准数据。

量化后的速度是否提高取决于硬件和 kernel。低比特 ≠ 必然更快。

当前 ONNX Runtime 的量化工具仍提供 `quantize_dynamic()` 与 `quantize_static()`。

最小动态量化接口：

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    "model.onnx",
    "model_int8.onnx",
    weight_type=QuantType.QInt8,
)
```

真实部署必须再做精度回归测试。

## 11.3.3 混合精度

为什么不全部 FP16？

部分算子可能：

- 对数值范围敏感；
- FP16 kernel 不合适；
- 精度下降超标。

混合精度允许：

- 大部分图使用 FP16；
- 少数敏感算子继续 FP32。

本质是速度 / 显存 / 精度三者的工程折中。

## 11.3.4 计算图优化

原教程列举：

- Identity elimination
- Dropout elimination
- Conv + BatchNorm fusion
- Conv + Add fusion
- Reshape fusion
- layout optimization

可以抽象为两类：

### 消除

删掉推理时没有意义的节点。

### 融合

把多个算子合成一个更高效 kernel。

例如：

```text
Conv → BatchNorm
```

可能融合为：

```text
Fused Conv
```

In [ ]:
# 纯数学演示：推理阶段 Linear 后接恒等操作没有任何意义
x = torch.randn(3, 4)
linear = nn.Linear(4, 4).eval()

with torch.inference_mode():
    y1 = linear(x)
    y2 = torch.clone(y1)  # 可以视作一种无意义中间操作的直觉演示

print("same values:", torch.allclose(y1, y2))

## ONNX Runtime Graph Optimization Level

常见：

- `ORT_DISABLE_ALL`
- `ORT_ENABLE_BASIC`
- `ORT_ENABLE_EXTENDED`
- `ORT_ENABLE_ALL`

不是级别越高就一定对所有模型收益越大，仍需实测。

In [ ]:
if HAS_ONNX:
    levels = [
        "ORT_DISABLE_ALL",
        "ORT_ENABLE_BASIC",
        "ORT_ENABLE_EXTENDED",
        "ORT_ENABLE_ALL",
    ]
    for name in levels:
        print(name, "->", getattr(ort.GraphOptimizationLevel, name))
else:
    print("SKIP")

## 11.3.5 SessionOptions：把运行策略和模型逻辑分离

示例：

```python
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)

session = ort.InferenceSession(
    "model.onnx",
    sess_options=sess_options,
    providers=["CPUExecutionProvider"],
)
```

模型文件没变，但 runtime 执行策略可以变化。

In [ ]:
if HAS_ONNX and dynamic_onnx_path.exists():
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )

    optimized_session = ort.InferenceSession(
        str(dynamic_onnx_path),
        sess_options=sess_options,
        providers=["CPUExecutionProvider"],
    )

    print("optimized session created")
else:
    optimized_session = None
    print("SKIP")

## 11.3.6 CPU 线程管理

两个高频参数：

- `intra_op_num_threads`：单个算子内部并行线程；
- `inter_op_num_threads`：不同图节点并行的线程。

还有 execution mode：

- `ORT_SEQUENTIAL`
- `ORT_PARALLEL`

注意：

> 更多线程不等于更快。

线程过多会导致调度、缓存竞争与 oversubscription。

In [ ]:
if HAS_ONNX:
    thread_options = ort.SessionOptions()
    thread_options.intra_op_num_threads = 2
    thread_options.inter_op_num_threads = 1
    thread_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

    print("intra:", thread_options.intra_op_num_threads)
    print("inter:", thread_options.inter_op_num_threads)
    print("mode:", thread_options.execution_mode)
else:
    print("SKIP")

## 11.3.7 I/O Binding：优化的不是算子，而是数据搬运

默认 GPU 推理时常见路径：

```text
NumPy on CPU
↓ H2D copy
GPU inference
↓ D2H copy
NumPy on CPU
```

如果上游 / 下游数据本来就在 GPU：

```text
GPU tensor
↓
ORT GPU
↓
GPU output
```

就没有必要为了调用 runtime 强制绕回 CPU。

I/O Binding 的作用就是提前绑定输入输出所在设备，减少不必要的数据复制。

当前 ONNX Runtime 的典型形式：

```python
io_binding = session.io_binding()
io_binding.bind_ortvalue_input("input", input_ortvalue)
io_binding.bind_ortvalue_output("output", output_ortvalue)

session.run_with_iobinding(io_binding)
```

是否有显著收益取决于：

- 输入输出张量大小；
- 是否真的在 GPU；
- 数据是否被后续 GPU 算子继续消费；
- H2D/D2H 在端到端延迟中的占比。

CPU-only 小模型一般看不到优势。

## 11.3.8 Profiling：不要猜性能瓶颈

ORT 可输出运行时 profile：

```python
options = ort.SessionOptions()
options.enable_profiling = True

session = ort.InferenceSession(
    "model.onnx",
    sess_options=options,
    providers=[...],
)

session.run(...)
profile_path = session.end_profiling()
```

profile 中可以看到：

- node 名称；
- duration；
- thread；
- execution provider；
- shape / type 等事件信息。

优化部署时：

> 先 profile，再优化。

In [ ]:
if HAS_ONNX and dynamic_onnx_path.exists():
    profile_options = ort.SessionOptions()
    profile_options.enable_profiling = True

    profile_session = ort.InferenceSession(
        str(dynamic_onnx_path),
        sess_options=profile_options,
        providers=["CPUExecutionProvider"],
    )

    x = np.random.randn(4, 4).astype(np.float32)
    _ = profile_session.run(None, {"input": x})

    profile_path = profile_session.end_profiling()
    print("profile file:", profile_path)
else:
    print("SKIP")

# 11.3.9 部署优化的正确顺序

不要一上来同时打开 FP16、INT8、图融合、线程调优和 I/O Binding。

更可靠的实验流程：

```text
1. PyTorch baseline
2. ONNX export
3. 数值正确性验证
4. ORT baseline benchmark
5. 单独改变一个优化变量
6. 再验证精度
7. 再 benchmark
8. profile 找剩余瓶颈
```

否则一旦结果异常，很难判断是哪一层导致。

# ONNX 部署检查清单

## 导出前

- `model.eval()`
- 明确输入 dtype / shape
- 明确哪些维度必须动态
- 确认目标 backend 支持的 opset / operators

## 导出后

- `onnx.checker.check_model`
- 查看 input / output name
- 查看 symbolic dimensions
- PyTorch 与 ORT 做数值对齐

## 性能测试

- warm-up
- latency
- throughput
- batch size
- provider
- dtype
- threads
- 是否包含数据复制

## 优化后

- 再做 accuracy / numerical regression
- 再 benchmark
- 不根据文件大小直接推断推理速度

# 章末知识结构总结

## 1. ONNX

负责**描述模型计算图**：

`PyTorch Module → ONNX Graph`

核心概念：

- graph
- node
- operator
- input / output
- initializer
- opset
- dynamic shape

## 2. ONNX Runtime

负责**执行计算图**：

`ONNX Graph → Execution Provider → Hardware`

核心：

- `InferenceSession`
- `session.run`
- providers
- SessionOptions

## 3. 正确性

部署第一优先级不是加速，而是：

`PyTorch output ≈ ORT output`

## 4. 性能

必须同时理解：

- latency
- throughput
- batch size

## 5. 优化层级

- FP16 / INT8：数值表示
- graph optimization：计算逻辑
- threads：CPU 调度
- I/O Binding：数据传输
- profiling：性能诊断

这五层不要混为一谈。

# 学完必须会回答的 12 个问题

1. ONNX 为什么能把框架适配复杂度从近似 `O(NM)` 降到 `O(N+M)`？
2. ONNX 文件里的 Graph、Node、Operator、Initializer 分别是什么？
3. ONNX package 与 ONNX Runtime 有什么区别？
4. opset version 为什么不能简单认为“越新越好”？
5. 当前 PyTorch 为什么更推荐 `dynamo=True` 的 ONNX exporter？
6. `dynamic_shapes` 解决什么问题？它与旧 `dynamic_axes` 的关系是什么？
7. 为什么导出 `.onnx` 成功后还必须做 PyTorch / ORT 数值对齐？
8. Execution Provider 是什么？为什么 provider 顺序有意义？
9. latency 与 throughput 有什么区别？为什么增大 batch 往往提高吞吐但可能提高延迟？
10. dynamic quantization 与 static quantization 最核心的区别是什么？
11. I/O Binding 优化的是模型计算还是 CPU/GPU 数据搬运？
12. 为什么模型部署优化应该遵循“baseline → 单变量修改 → 正确性验证 → benchmark → profiling”？

# 面向后续 TensorRT / LLM 部署的复习优先级

如果下一章继续学习 TensorRT，优先掌握：

1. **PyTorch → ONNX 计算图**
2. **operator / opset**
3. **dynamic shapes**
4. **PyTorch vs ORT 数值对齐**
5. **latency / throughput / batch**
6. **FP16 / INT8 基础**
7. **graph optimization**
8. **I/O Binding**
9. **profiling**

对于 LLM 部署，还要额外关注：

- dynamic sequence length
- KV Cache 输入输出
- large external weights
- attention operator compatibility
- quantization strategy